# Независимые фильтры после геометрии

Notebook подготовлен без выполнения. По умолчанию читаются сохранённые геометрически прошедшие кандидаты всех семи структур. Старый результат Li–Li не ограничивает вход. Поиск и геометрия повторно не запускаются.

Новая схема Вороного имеет рабочие, ещё не откалиброванные допуски. Численности и координаты появятся только после отдельно разрешённого запуска. Старые отчёты не являются результатами этой реализации.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dataclasses import replace
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "elec_voids_search").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
from elec_voids_search.voronoi_coordination_config import VoronoiCoordinationConfig
from elec_voids_search.probe_separation import ProbeSeparationConfig
from elec_voids_search.post_geometry_config import PostGeometryConfig
from elec_voids_search.screening_pipeline import continue_from_saved_geometry

BASELINE = PROJECT_ROOT / "elec_voids_results/pipeline_compact_20261024_200915"
manual = json.loads((BASELINE / "run_metadata.json").read_text())["charges"]


## Видимые настройки и последовательности

`paths` задаёт независимые маршруты от общего геометрического входа. Ключ — имя маршрута, значение — порядок фильтров. Внутри последовательности только прошедшие точки поступают дальше. Отклонённые и неопределённые сохраняются в результатах шага. Повтор одного фильтра внутри маршрута запрещён.

CIF выключены по умолчанию. При `export_cif=True` записываются прошедшие и отклонённые точки каждого шага маршрута; пустая группа даёт CIF с одним каркасом.


In [ ]:
COORDINATION = VoronoiCoordinationConfig()
SEPARATION = ProbeSeparationConfig(probe_element="Li", distance_tolerance_A=0.10)
PATHS = {
    "coordination": ("coordination",),
    "li_distance": ("li_distance",),
}
# Для проверки последовательностей можно добавить или заменить маршруты:
# PATHS["coord_then_li"] = ("coordination", "li_distance")
# PATHS["li_then_coord"] = ("li_distance", "coordination")

# Все пороги меняются явно, например:
# COORDINATION = replace(COORDINATION, distance_tolerance_A=0.25,
#                        angular_tolerance_deg=15.0, combined_score_limit=1.0)
# SEPARATION = replace(SEPARATION, distance_tolerance_A=0.15)

SETTINGS = PostGeometryConfig(
    probe_element="Li", probe_oxidation_state=1,
    coordination=COORDINATION, separation=SEPARATION, paths=PATHS,
    export_cif=False,
)
parameter_rows = []
for block, values in SETTINGS.to_dict().items():
    if isinstance(values, dict) and block != "paths":
        parameter_rows.extend({"block": block, "parameter": key, "value": repr(value)}
                              for key, value in values.items())
    else:
        parameter_rows.append({"block": "pipeline", "parameter": block, "value": repr(values)})
display(pd.DataFrame(parameter_rows))


## Семь тестовых структур

LATP — 43, LLZO–Ga — 27, LGPS — 5, обе искусственные вакансии — по 3, исходные LiCl и Li3PO4 — по 0 согласно базовой геометрической сводке. Это размеры входных выборок, а не новые результаты Вороного.


In [ ]:
baseline_summary = pd.read_csv(BASELINE / "summary.csv")
inputs = []
for _, row in baseline_summary.iterrows():
    name = Path(row.structure_file).stem
    input_folder = "cif_tests_2" if "_one_Li_vacancy" in name else "kobernik_voids/cifs"
    inputs.append({"structure": name,
                   "input_cif": PROJECT_ROOT / input_folder / row.structure_file,
                   "saved_candidates": BASELINE / name / "candidates.csv",
                   "n_geometry_input": int(row.n_geometry_passed)})
display(pd.DataFrame(inputs))


## Продолжение отбора

Сохраняются одна таблица кандидатов с результатами веток, краткая сводка, настройки/происхождение и таблицы занятых эталонов. Полные сырые стадии и большие JSON с координатами повторно не выгружаются. Прежние каталоги и базовая версия не изменяются.


In [ ]:
stamp = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = PROJECT_ROOT / "elec_voids_results" / f"weighted_voronoi_branches_{stamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
results, summaries, failures = {}, [], []
for item in inputs:
    name = item["structure"]
    print(name, flush=True)
    try:
        result = continue_from_saved_geometry(
            item["input_cif"], item["saved_candidates"],
            oxidation_states=manual[name], config=SETTINGS,
            output_dir=OUTPUT_DIR / name,
        )
    except Exception as error:
        failures.append({"structure": name, "error": repr(error)})
        print("Ошибка:", repr(error), flush=True)
        continue
    results[name] = result
    summary = result["summary"].copy()
    summary.insert(0, "structure", name)
    summaries.append(summary)
all_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
all_summary.to_csv(OUTPUT_DIR / "all_structures_summary.csv", index=False)
if failures:
    pd.DataFrame(failures).to_csv(OUTPUT_DIR / "failures.csv", index=False)
display(all_summary)
display(pd.DataFrame(failures, columns=["structure", "error"]))
print("Результаты:", OUTPUT_DIR)


## Координаты и причины решений

`filter_comparison_group`: `passed_both`, `coordination_only`, `li_distance_only`, `rejected_both`, `indeterminate` либо `not_compared_by_both`. Последняя группа возникает, когда при выбранной последовательности часть точек не дошла до второго фильтра. Это не отказ второго фильтра.

Блоковые оценки нормированы на выбранные допуски; настройки показываются выше. Ошибка построения многогранника или неполный набор эталонов не маскируются геометрическим отказом.


In [ ]:
columns = ["candidate_id", "frac_x", "frac_y", "frac_z",
           "coordination_status", "probe_distance_status", "filter_comparison_group",
           "coordination_reference_site_index", "coordination_reason",
           "coordination_score", "coordination_distance_score", "coordination_size_score",
           "coordination_shape_score", "coordination_combined_score",
           "coordination_primary_score"]
for name, result in results.items():
    print(name)
    display(result["candidates"][[column for column in columns if column in result["candidates"]]])
    for branch_name, branch in result["branches"].items():
        print(branch_name, "прошли:", branch["passed"].candidate_id.tolist(),
              "отклонены:", branch["rejected"].candidate_id.tolist(),
              "неопределённо:", branch["indeterminate"].candidate_id.tolist())


## Отдельная проверка следующего фильтра

Можно передать `results[name]["branches"]["coordination"]["passed"]` в `screen_geometry_candidates` с одним маршрутом `("li_distance",)`, либо наоборот. Для одновременной проверки обеих последовательностей удобнее добавить два маршрута в PATHS: общие оценки будут переиспользованы внутри одного запуска.

Подготовленные модульные тесты находятся в tests/test_post_geometry.py. Они не выполняются автоматически этим notebook. Пример отдельного запуска после разрешения: `python -m unittest discover -s elec_voids_search/tests`.
